In [ ]:
import os
import random

import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()


HDF5_PATH = "/scratch/gpfs/MLISANTI/ra0438/all_bursts.hdf5"
CATALOG_PATH = "/home/ra0438/chime-catalog-two-analysis/chimefrbcat2.csv"
CHECKPOINT_DIR = "/scratch/gpfs/MLISANTI/ra0438/resnet_ae_checkpoints_cv_holdout"

TARGET_LENGTH = 128       # number of time samples kept per burst window
N_FREQ_CHANNELS = 256     # number of frequency channels in the waterfall

N_SPLITS = 5
BATCH_SIZE = 64
NUM_WORKERS = 5
TRAIN_FRAC = 0.7          # only affects the (unused here) Optuna train/val split
HOLDOUT_FRAC = 0.15


class CHIMEFRBDataset(Dataset):
    def __init__(self, hdf5_path, catalog_path, target_length):
        self.hdf5_path = hdf5_path
        self.dt = 0.009830400085775182 * 1000
        self._h5 = None

        self.target_length = target_length
        cat = pd.read_csv(catalog_path, low_memory=False)
        cat["repeater_name"] = cat["repeater_name"].fillna("").str.strip()
        self.repeater_set = set(
            cat.loc[cat["repeater_name"] != "", "tns_name"].str.strip()
        )
        with h5py.File(hdf5_path, "r") as f:
            self.keys = list(f.keys())
        self.labels = np.array([int(k in self.repeater_set) for k in self.keys], dtype=np.int64)

    @staticmethod
    def _pad_or_crop(wfall, target_length, center_idx):
        n_freq, n_time = wfall.shape
        half = target_length // 2
        start = center_idx - half
        end = start + target_length

        if start >= 0 and end <= n_time:
            return wfall[:, start:end]

        src_start = max(start, 0)
        src_end = min(end, n_time)
        out = np.zeros((n_freq, target_length), dtype=wfall.dtype)
        dst_start = src_start - start
        out[:, dst_start:dst_start + (src_end - src_start)] = wfall[:, src_start:src_end]
        return out

    def _get_file(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.hdf5_path, "r")
        return self._h5

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        key = self.keys[idx]

        f = self._get_file()
        wfall = f[key]["wfall_plot"][:]
        extent = np.array(f[key]["extent"])

        wfall = wfall.astype(np.float32)
        std = wfall.std(axis=1, keepdims=True)
        std[std == 0] = 1.0  # avoid divide-by-zero for masked channels
        wfall = (wfall - wfall.mean(axis=1, keepdims=True)) / std

        peak = round(-extent[0] / self.dt)
        wfall = self._pad_or_crop(wfall, self.target_length, peak)
        tensor = torch.from_numpy(np.ascontiguousarray(wfall))
        label = torch.tensor(int(key in self.repeater_set), dtype=torch.long)

        return tensor, label


def make_loader(dataset, idx, shuffle=False, drop_last=False):
    return DataLoader(Subset(dataset, idx), batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), drop_last=drop_last)


def split_indices(labels, train_frac=TRAIN_FRAC, holdout_frac=HOLDOUT_FRAC, seed=SEED):
    """Same splits as the training script: returns (CV pool indices, holdout indices)."""
    indices = list(range(len(labels)))
    trainval_idx, holdout_idx = train_test_split(
        indices, test_size=holdout_frac, stratify=labels, random_state=seed
    )
    return np.array(trainval_idx), np.array(holdout_idx)


# ----------------------------------------------------------------------------
# Model: ResNet conv encoder -> bottleneck -> (ResNet conv decoder, classifier head)
# Identical to the training script so the checkpoints load.
# ----------------------------------------------------------------------------
class ResBlock(nn.Module):
    """Pre-activation basic residual block (BN-ReLU-Conv x2) with optional stride."""
    def __init__(self, in_ch, out_ch, stride=1, dropout=0.0):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.drop = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)

        self.shortcut = None
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        sc = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(self.drop(F.relu(self.bn2(out))))
        return out + sc


class UpBlock(nn.Module):
    """2x upsample (transposed conv) followed by residual blocks."""
    def __init__(self, in_ch, out_ch, n_blocks, dropout=0.0):
        super().__init__()
        self.up = nn.Sequential(nn.BatchNorm2d(in_ch), nn.ReLU(inplace=True),
                                nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2, bias=False))
        self.blocks = nn.Sequential(*[ResBlock(out_ch, out_ch, dropout=dropout) for _ in range(n_blocks)])

    def forward(self, x):
        return self.blocks(self.up(x))


class FRBResNetAE(nn.Module):
    def __init__(self, n_freq, n_time, base_channels=32, n_stages=3, enc_blocks=2, dec_blocks=1,
                 max_channels=256, latent_channels=None, embed_dim=64, dropout=0.1, head_hidden=128,
                 mask_ratio=0.0, mask_patch_f=16, mask_patch_t=8):
        super().__init__()
        down = 2 ** n_stages
        assert n_freq % down == 0 and n_time % down == 0, "n_freq/n_time must be divisible by 2**n_stages"
        assert n_freq % mask_patch_f == 0 and n_time % mask_patch_t == 0

        self.n_freq, self.n_time = n_freq, n_time
        self.mask_ratio = mask_ratio
        self.mask_patch_f, self.mask_patch_t = mask_patch_f, mask_patch_t

        chs = [min(base_channels * 2 ** i, max_channels) for i in range(n_stages + 1)]
        latent_channels = latent_channels or chs[-1]

        # ---- encoder ----
        self.stem = nn.Conv2d(1, chs[0], 3, padding=1, bias=False)
        enc = []
        for i in range(n_stages):
            stage = [ResBlock(chs[i], chs[i + 1], stride=2, dropout=dropout)]
            stage += [ResBlock(chs[i + 1], chs[i + 1], dropout=dropout) for _ in range(enc_blocks - 1)]
            enc.append(nn.Sequential(*stage))
        self.enc_stages = nn.Sequential(*enc)
        self.to_latent = nn.Sequential(nn.BatchNorm2d(chs[-1]), nn.ReLU(inplace=True),
                                       nn.Conv2d(chs[-1], latent_channels, 1))

        # ---- decoder ----
        self.from_latent = nn.Conv2d(latent_channels, chs[-1], 1)
        dec = [UpBlock(chs[i + 1], chs[i], dec_blocks, dropout=dropout) for i in reversed(range(n_stages))]
        self.dec_stages = nn.Sequential(*dec)
        self.dec_out = nn.Sequential(nn.BatchNorm2d(chs[0]), nn.ReLU(inplace=True),
                                     nn.Conv2d(chs[0], 1, 3, padding=1))

        # ---- embedding (mean-pooled latent -> vector) ----
        self.embed_head = nn.Sequential(nn.Linear(latent_channels, latent_channels),
                                        nn.LayerNorm(latent_channels),
                                        nn.ReLU(),
                                        nn.Linear(latent_channels, embed_dim))

        # ---- classifier head (on the embedding) ----
        self.cls_head = nn.Sequential(nn.Linear(embed_dim, head_hidden),
                                      nn.LayerNorm(head_hidden),
                                      nn.ReLU(),
                                      nn.Dropout(dropout),
                                      nn.Linear(head_hidden, head_hidden // 2),
                                      nn.ReLU(),
                                      nn.Dropout(dropout),
                                      nn.Linear(head_hidden // 2, 1))

        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)

    def random_patch_mask(self, x):
        """Returns pixel mask [B, F, T] with 1 = hidden, hiding exactly mask_ratio of the patches."""
        B = x.size(0)
        gf, gt = self.n_freq // self.mask_patch_f, self.n_time // self.mask_patch_t
        n = gf * gt
        n_mask = int(round(n * self.mask_ratio))
        noise = torch.rand(B, n, device=x.device)
        ids = torch.argsort(noise, dim=1)[:, :n_mask]
        m = torch.zeros(B, n, device=x.device)
        m.scatter_(1, ids, 1.0)
        m = m.view(B, gf, gt)
        m = m.repeat_interleave(self.mask_patch_f, dim=1).repeat_interleave(self.mask_patch_t, dim=2)
        return m

    def encode(self, x):
        h = self.stem(x.unsqueeze(1))                # [B, C0, F, T]
        h = self.enc_stages(h)                       # [B, C_n, F/2^n, T/2^n]
        return self.to_latent(h)                     # [B, L, F/2^n, T/2^n]

    def decode(self, z):
        h = self.dec_stages(self.from_latent(z))
        return self.dec_out(h).squeeze(1)            # [B, F, T]

    def embed(self, z):
        return self.embed_head(z.mean(dim=(2, 3)))   # [B, embed_dim]

    def classify(self, e):
        return self.cls_head(e).squeeze(-1)          # [B]

    def forward(self, x, apply_mask=None):
        if apply_mask is None:
            apply_mask = self.training
        mask = None
        x_in = x
        if apply_mask and self.mask_ratio > 0:
            mask = self.random_patch_mask(x)
            x_in = x * (1 - mask)
        z = self.encode(x_in)
        e = self.embed(z)
        return self.classify(e), self.decode(z), mask, e


def build_model_from_params(p):
    mask_patch_f, mask_patch_t = map(int, p["mask_patch"].split("x"))
    return FRBResNetAE(n_freq=N_FREQ_CHANNELS, n_time=TARGET_LENGTH,
                       base_channels=p["base_channels"], n_stages=p["n_stages"],
                       enc_blocks=p["enc_blocks"], dec_blocks=p["dec_blocks"],
                       max_channels=p["max_channels"], latent_channels=p["latent_channels"],
                       embed_dim=p["embed_dim"], dropout=p["dropout"], head_hidden=p["head_hidden"],
                       mask_ratio=p["mask_ratio"], mask_patch_f=mask_patch_f,
                       mask_patch_t=mask_patch_t).to(device)


# ----------------------------------------------------------------------------
# Classifier-only evaluation helpers
# ----------------------------------------------------------------------------
@torch.no_grad()
def get_probs_and_labels(model, loader, device):
    """Clean (unmasked) forward through encoder -> embedding -> classifier. The decoder is skipped."""
    model.eval()
    all_probs, all_labels = [], []
    for wfall, labels in loader:
        wfall = wfall.to(device)
        logits = model.classify(model.embed(model.encode(wfall)))
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)


def balanced_accuracy_from_cm(cm):
    tn, fp, fn, tp = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    tnr = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return 0.5 * (tpr + tnr)


def sweep_f1_score(all_probs, all_labels):
    best_thresh, best_f1 = 0.5, -1.0
    best_cm = None

    for thresh in np.linspace(0.05, 0.95, 91):
        preds = (all_probs > thresh).astype(int)
        cm = confusion_matrix(all_labels, preds, labels=[0, 1])
        f1 = f1_score(all_labels, preds, pos_label=1, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh
            best_cm = cm

    best_preds = (all_probs > best_thresh).astype(int)
    acc = (best_preds == all_labels).mean()
    balacc = balanced_accuracy_from_cm(best_cm)

    return best_thresh, balacc, acc, best_f1, best_cm


def metrics_at_threshold(probs, labels, thresh):
    preds = (probs > thresh).astype(int)
    cm = confusion_matrix(labels, preds, labels=[0, 1])
    return (
        balanced_accuracy_from_cm(cm),
        (preds == labels).mean(),
        f1_score(labels, preds, pos_label=1, zero_division=0),
        cm,
    )


def pooled_stats(cm):
    tn, fp, fn, tp = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return balanced_accuracy_from_cm(cm), (tn + tp) / cm.sum(), prec, rec, f1


# ----------------------------------------------------------------------------
# Data + splits (same seeds as training)
# ----------------------------------------------------------------------------
dataset = CHIMEFRBDataset(HDF5_PATH, CATALOG_PATH, TARGET_LENGTH)
cv_idx, holdout_idx = split_indices(dataset.labels)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

holdout_loader = make_loader(dataset, holdout_idx)
n_rep_holdout = int(dataset.labels[holdout_idx].sum())
print(f"Dataset: {len(dataset)} | CV pool: {len(cv_idx)} | Holdout: {len(holdout_idx)}")
print(f"Holdout: {len(holdout_idx)} | repeaters: {n_rep_holdout} ({100 * n_rep_holdout / len(holdout_idx):.1f}%)")


# ----------------------------------------------------------------------------
# Per-fold evaluation: out-of-fold validation, then holdout at the val-derived threshold
# ----------------------------------------------------------------------------
global_cm = np.zeros((2, 2), dtype=int)
global_holdout_cm = np.zeros((2, 2), dtype=int)
fold_results = []

for fold_idx, (train_pos, val_pos) in enumerate(skf.split(cv_idx, dataset.labels[cv_idx])):
    val_idx = cv_idx[val_pos]

    ckpt_path = os.path.join(CHECKPOINT_DIR, f"fold_{fold_idx}.pt")
    if not os.path.exists(ckpt_path):
        print(f"Fold {fold_idx}: checkpoint not found at {ckpt_path}, skipping.")
        continue

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = build_model_from_params(ckpt["params"])
    model.load_state_dict(ckpt["model_state_dict"])

    val_loader_fold = make_loader(dataset, val_idx)
    all_probs, all_labels = get_probs_and_labels(model, val_loader_fold, device)

    thresh, balacc, acc, f1, cm = sweep_f1_score(all_probs, all_labels)
    global_cm += cm

    n_rep_val = int(dataset.labels[val_idx].sum())
    print(f"\n===== Fold {fold_idx + 1}/{N_SPLITS} =====")
    print(f"Val: {len(val_idx)} | repeaters: {n_rep_val} ({100 * n_rep_val / len(val_idx):.1f}%)")
    print(f"Best threshold (by f1): {thresh:.2f}   "
          f"[checkpoint: F1={ckpt['val_f1']:.4f} @ {ckpt['best_thresh']:.2f}, epoch {ckpt['best_epoch']}]")
    print(f"Balanced accuracy : {balacc:.4f}")
    print(f"Accuracy          : {acc:.4f}")
    print(f"F1                : {f1:.4f}")
    print("Confusion matrix:")
    print(cm)

    h_probs, h_labels = get_probs_and_labels(model, holdout_loader, device)
    h_balacc, h_acc, h_f1, h_cm = metrics_at_threshold(h_probs, h_labels, thresh)
    global_holdout_cm += h_cm

    print(f"--- Holdout (threshold fixed from val = {thresh:.2f}) ---")
    print(f"Balanced accuracy : {h_balacc:.4f}")
    print(f"Accuracy          : {h_acc:.4f}")
    print(f"F1                : {h_f1:.4f}")
    print("Confusion matrix:")
    print(h_cm)

    fold_results.append(
        {
            "fold": fold_idx,
            "threshold": thresh,
            "balanced_accuracy": balacc,
            "accuracy": acc,
            "f1": f1,
            "confusion_matrix": cm,
            "holdout_balanced_accuracy": h_balacc,
            "holdout_accuracy": h_acc,
            "holdout_f1": h_f1,
            "holdout_confusion_matrix": h_cm,
        }
    )


# ----------------------------------------------------------------------------
# Summaries
# ----------------------------------------------------------------------------
balaccs = np.array([r["balanced_accuracy"] for r in fold_results])
accs = np.array([r["accuracy"] for r in fold_results])
f1s = np.array([r["f1"] for r in fold_results])

print("===== Per-Fold Summary =====")
for r in fold_results:
    print(f"Fold {r['fold'] + 1}: BalAcc={r['balanced_accuracy']:.4f}  "
          f"Accuracy={r['accuracy']:.4f}  F1={r['f1']:.4f}  thresh={r['threshold']:.2f}")

print(f"\nMean Balanced Accuracy : {balaccs.mean():.4f} +/- {balaccs.std():.4f}")
print(f"Mean Accuracy          : {accs.mean():.4f} +/- {accs.std():.4f}")
print(f"Mean F1                : {f1s.mean():.4f} +/- {f1s.std():.4f}")

g_balacc, g_acc, g_prec, g_rec, g_f1 = pooled_stats(global_cm)
print("\n===== Global (Pooled Out-of-Fold) Confusion Matrix =====")
print(global_cm)
print(f"Global Balanced Accuracy : {g_balacc:.4f}")
print(f"Global Accuracy          : {g_acc:.4f}")
print(f"Global Precision         : {g_prec:.4f}")
print(f"Global Recall            : {g_rec:.4f}")
print(f"Global F1                : {g_f1:.4f}")

h_balaccs = np.array([r["holdout_balanced_accuracy"] for r in fold_results])
h_accs = np.array([r["holdout_accuracy"] for r in fold_results])
h_f1s = np.array([r["holdout_f1"] for r in fold_results])

print("\n===== Per-Fold Holdout Summary =====")
for r in fold_results:
    print(f"Fold {r['fold'] + 1}: BalAcc={r['holdout_balanced_accuracy']:.4f}  "
          f"Accuracy={r['holdout_accuracy']:.4f}  F1={r['holdout_f1']:.4f}  "
          f"thresh(from val)={r['threshold']:.2f}")

print(f"\nMean Holdout Balanced Accuracy : {h_balaccs.mean():.4f} +/- {h_balaccs.std():.4f}")
print(f"Mean Holdout Accuracy          : {h_accs.mean():.4f} +/- {h_accs.std():.4f}")
print(f"Mean Holdout F1                : {h_f1s.mean():.4f} +/- {h_f1s.std():.4f}")

hb, ha, hp, hr, hf = pooled_stats(global_holdout_cm)
print("\n===== Global (Pooled Across Fold Models) Holdout Confusion Matrix =====")
print(global_holdout_cm)
print(f"Global Holdout Balanced Accuracy : {hb:.4f}")
print(f"Global Holdout Accuracy          : {ha:.4f}")
print(f"Global Holdout Precision         : {hp:.4f}")
print(f"Global Holdout Recall            : {hr:.4f}")
print(f"Global Holdout F1                : {hf:.4f}")